# Task 2: RAG from scratch (Kaggle GPU + Ollama)

PDF → word chunks → embeddings (Ollama) → NumPy cosine search (+ optional BM25/RRF hybrid) → LLM answer.
No LangChain / LlamaIndex. Compares two chunk sizes by recall@5, equal-budget recall and MRR.

**Before you run anything**
1. Notebook settings → **Accelerator: GPU (T4 or P100)** and **Internet: On**.
2. Upload your public PDF (30+ pages, real text, not scans) and `questions.json` (15 questions *you* wrote), or set `PDF_URL` below.
3. Run the cells top to bottom. Cells 5 to 7 are the only ones that need your files.

`questions.json` format: `[{"question": "...", "passage": "1-2 sentences copied from the PDF text", "page": 12}, ...]`

## 1. Config

In [56]:
LLM_MODEL   = "qwen2.5:7b-instruct"   # fits a 16 GB T4/P100; if you hit out-of-memory, use "qwen2.5:3b-instruct"
EMBED_MODEL = "nomic-embed-text"
PDF_PATH = "llama2.pdf"
PDF_URL  = "https://arxiv.org/pdf/2307.09288.pdf"                 # optional: direct link to a public PDF (downloaded only if PDF_PATH is missing)
QUESTIONS   = "questions.json"
REQUIRE_GPU = True                    # stop early if the GPU is not enabled
CHUNK_SIZES = (100, 300)              # two chunk sizes in words

## 2. Environment check

In [57]:
import json, os, subprocess, sys
from pathlib import Path


def sh(cmd, check=True):
    # run a shell command and stream its output live into the cell
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    if check and rc:
        raise RuntimeError(f"command failed (exit {rc}): {cmd}")
    return rc


try:
    gpu = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True, stderr=subprocess.DEVNULL
    ).strip()
except Exception:
    gpu = None
print("GPU:", gpu or "none detected")
if REQUIRE_GPU and not gpu:
    raise RuntimeError("No GPU visible. Enable it under Settings > Accelerator, then restart the session.")
print("Python", sys.version.split()[0])

GPU: Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
Python 3.13.16


## 3. Install Ollama, start it, pull the models

In [58]:
!pip install -q pypdf numpy
Path('kaggle_setup.sh').write_text(r'''#!/usr/bin/env bash
# Install Ollama on a Kaggle notebook, start it, and pull one embedding model + one LLM.
# Notebook settings: Accelerator = GPU (T4 / P100), Internet = ON.
# Usage in a cell:   !bash kaggle_setup.sh qwen2.5:7b-instruct nomic-embed-text
set -u
LLM="${1:-qwen2.5:7b-instruct}"
EMBED="${2:-nomic-embed-text}"

if ! nvidia-smi >/dev/null 2>&1; then
  echo "WARNING: no GPU visible. Enable it in Kaggle: Settings > Accelerator. Ollama will fall back to CPU (slow)."
fi

apt-get install -y -qq zstd pciutils lshw >/dev/null 2>&1 || true   # the Ollama installer needs zstd
if ! command -v ollama >/dev/null 2>&1; then
  curl -fsSL https://ollama.com/install.sh | sh || { echo "Ollama install failed (is Internet ON?)"; exit 1; }
fi

nohup ollama serve > ollama.log 2>&1 &
for _ in $(seq 1 40); do
  curl -s http://localhost:11434/api/tags >/dev/null 2>&1 && break
  sleep 1
done
curl -s http://localhost:11434/api/tags >/dev/null 2>&1 || { echo "Ollama did not start; see ollama.log"; exit 1; }

ollama pull "$EMBED" && ollama pull "$LLM" || { echo "Model pull failed; check the exact tag on ollama.com/library"; exit 1; }
echo "READY: embeddings=$EMBED  llm=$LLM"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null
''', encoding='utf-8')
sh(f"bash kaggle_setup.sh {LLM_MODEL} {EMBED_MODEL}")

pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕██████████████████▏   17 B                         
pulling 31df23ea7daa: 100% ▕██████████████████▏  420 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success 
pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling 2bada8a74506: 100% ▕██████████████████▏ 4.7 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling eb4402837c78: 100% ▕██████████████████▏ 1.5 KB                         
pulling 832dd9e00a68: 100% ▕██████████████████▏  11 KB                         
pulling 2f15b3218f05: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success 
READY: embedding

0

In [59]:
sh("pip install -q pypdf numpy")

0

## 4. The pipeline (`rag.py`, tested with 18 edge cases)
This cell writes the script; it does not run it.

In [60]:
Path('rag.py').write_text(r'''"""
TASK 2 - RAG from scratch: PDF -> chunks -> Ollama embeddings -> NumPy cosine search -> LLM answer.
No LangChain / LlamaIndex. Compares two chunk sizes by recall@k.

  Runs fully locally (Ollama on your machine or a Kaggle GPU notebook): no cloud API, no API key.
  ollama pull nomic-embed-text && ollama pull qwen2.5:7b-instruct
  pip install pypdf numpy
  python rag.py --pdf doc.pdf --questions questions.json --chunk-sizes 100 300 --answer
  add --hybrid to also score BM25 + dense (RRF) as a bonus; the dense-only results are the required ones.

questions.json: [{"question": "...", "passage": "text copied from the PDF", "page": 12}, ...]
"""
import argparse
import hashlib
import json
import math
import os
import re
import sys
import time
import urllib.error
import urllib.request
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from pypdf import PdfReader

NGRAM = 4  # hit rule works on word 4-grams


# ---------- Ollama client ----------
def post(args, path, payload, retries=3):
    """POST JSON to the local Ollama server."""
    headers = {"Content-Type": "application/json"}
    req = urllib.request.Request(
        args.base_url.rstrip("/") + path,
        json.dumps(payload).encode("utf-8"),
        headers,
        method="POST",
    )
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(req, timeout=300) as r:
                return json.loads(r.read().decode("utf-8"))
        except urllib.error.HTTPError as e:
            if e.code in {429, 500, 502, 503, 504} and attempt < retries - 1:
                time.sleep(2 ** attempt)
                continue
            raise RuntimeError(f"{path} failed ({e.code}): {e.read().decode(errors='replace')}") from e
        except (urllib.error.URLError, TimeoutError):
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)


def embed(args, texts, prefix="", batch=32):
    """L2-normalised embeddings (dot product == cosine), cached on disk by content hash."""
    key = hashlib.sha1(json.dumps([args.embed_model, prefix, texts]).encode("utf-8")).hexdigest()
    cache = Path(".cache") / f"{key}.npy"
    if cache.exists():
        return np.load(cache)
    vecs = []
    for i in range(0, len(texts), batch):
        out = post(
            args,
            "/api/embed",
            {"model": args.embed_model, "input": [prefix + t for t in texts[i : i + batch]]},
        )
        vecs.extend(out["embeddings"])
    arr = np.asarray(vecs, dtype="float32")
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    arr /= np.maximum(norms, 1e-12)
    cache.parent.mkdir(exist_ok=True)
    np.save(cache, arr)
    return arr


# ---------- documents and chunking ----------
@dataclass
class Chunk:
    text: str
    page: int
    page_end: int
    start: int

    @property
    def page_label(self):
        return f"page {self.page}" if self.page == self.page_end else f"pages {self.page}-{self.page_end}"


def load_pages(path):
    pages = []
    for i, p in enumerate(PdfReader(path).pages, 1):
        text = re.sub(r"(\w)-\n(\w)", r"\1\2", p.extract_text() or "")
        pages.append((i, text))
    empty = [i for i, t in pages if not t.strip()]
    print(f"Loaded {len(pages)} pages" + (f" (no text on pages {empty}: scanned? needs OCR)" if empty else ""))
    return pages


def make_chunks(pages, size, overlap, snap=True):
    """Sliding word windows. With snap, extends up to 20% to end cleanly on a sentence boundary."""
    words = [(w, pg) for pg, t in pages for w in t.split()]
    if not words:
        return []
    step = max(1, int(size * (1 - overlap)))
    chunks = []
    for s in range(0, len(words), step):
        end = min(s + size, len(words))
        if snap and end < len(words):
            limit = min(end + size // 5, len(words))
            for j in range(end - 1, limit):
                if words[j][0][-1] in ".?!":
                    end = j + 1
                    break
        chunks.append(Chunk(" ".join(w for w, _ in words[s:end]), words[s][1], words[end - 1][1], s))
        if end == len(words):
            break
    return chunks


# ---------- retrieval ----------
def top_k(scores, chunks, k, size, dedup):
    """Best k chunks by score; skips near-duplicate overlapping windows, then backfills up to k."""
    if not chunks:
        return []
    target_k = min(k, len(chunks))
    order = [int(i) for i in np.argsort(-scores)]
    picked = []
    for i in order:
        if dedup and any(abs(chunks[i].start - chunks[j].start) < size * 0.5 for j in picked):
            continue
        picked.append(i)
        if len(picked) == target_k:
            return picked
    if len(picked) < target_k:
        for i in order:
            if i not in picked:
                picked.append(i)
                if len(picked) == target_k:
                    break
    return picked


def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


class BM25:
    """Okapi BM25 with an inverted index. Rewards exact keyword matches weighted by term rarity."""

    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b, self.n = k1, b, len(docs)
        self.post = defaultdict(list)
        self.dl = np.zeros(self.n)
        for i, d in enumerate(docs):
            toks = tokenize(d)
            self.dl[i] = len(toks)
            for t, f in Counter(toks).items():
                self.post[t].append((i, f))
        self.avgdl = float(self.dl.mean()) if self.n > 0 else 1.0
        self.idf = {t: math.log(1 + (self.n - len(p) + 0.5) / (len(p) + 0.5)) for t, p in self.post.items()}

    def scores(self, query):
        s = np.zeros(self.n)
        if self.n == 0 or self.avgdl == 0:
            return s
        for t in set(tokenize(query)):
            for i, f in self.post.get(t, ()):
                denom = f + self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                s[i] += self.idf[t] * f * (self.k1 + 1) / denom
        return s


def rrf(dense_scores, bm25_scores, c=60):
    n = len(dense_scores)
    fused = np.zeros(n)

    for rank, idx in enumerate(np.argsort(-dense_scores)):
        fused[idx] += 1.0 / (c + rank + 1)

    for rank, idx in enumerate(np.argsort(-bm25_scores)):
        if bm25_scores[idx] <= 0:
            break
        fused[idx] += 1.0 / (c + rank + 1)

    return fused


# ---------- evaluation ----------
def grams(text, n=NGRAM):
    t = re.findall(r"[a-z0-9]+", text.lower())
    n = min(n, len(t)) or 1
    return {tuple(t[i : i + n]) for i in range(len(t) - n + 1)}


def coverage(gold, other):
    return len(gold & other) / len(gold) if gold else 0.0


def evaluate(args, pages, qs, q_vecs, size, mode="dense"):
    chunks = make_chunks(pages, size, args.overlap, not args.no_snap)
    c_vecs = embed(args, [c.text for c in chunks], args.doc_prefix)
    c_grams = [grams(c.text) for c in chunks]
    bm25 = BM25([c.text for c in chunks]) if mode == "hybrid" else None
    k_budget = max(1, args.budget // size)
    k_max = max(args.k, k_budget)
    hits_k = hits_b = 0
    rr = 0.0
    fails, picks_all = [], []
    for q, qv in zip(qs, q_vecs):
        scores = c_vecs @ qv
        if bm25:
            scores = rrf(scores, bm25.scores(q["question"]))
        picks = top_k(scores, chunks, k_max, size, not args.keep_overlaps)
        picks_all.append(picks)
        gold = grams(q["passage"])
        rank = next((r for r, i in enumerate(picks) if coverage(gold, c_grams[i]) >= args.threshold), None)
        hits_k += int(rank is not None and rank < args.k)
        hits_b += int(rank is not None and rank < k_budget)
        rr += 1.0 / (rank + 1) if rank is not None and rank < args.k else 0.0
        if rank is None or rank >= args.k:
            fails.append({
                "question": q["question"],
                "expected_page": q.get("page"),
                "retrieved_pages": [chunks[i].page for i in picks[: args.k]],
                "scores": [round(float(scores[i]), 3) for i in picks[: args.k]],
            })
    n = len(qs)
    print(
        f"{mode:>6} | chunk={size:>4} words | {len(chunks):>5} chunks | "
        f"recall@{args.k}={hits_k / n:.2f} ({hits_k}/{n}) | "
        f"equal-budget recall@{k_budget}={hits_b / n:.2f} | MRR@{args.k}={rr / n:.2f}"
    )
    return {
        "mode": mode,
        "chunk_size": size,
        "n_chunks": len(chunks),
        f"recall@{args.k}": round(hits_k / n, 3),
        f"recall@{k_budget}_equal_budget": round(hits_b / n, 3),
        f"mrr@{args.k}": round(rr / n, 3),
        "failures": fails,
        "_chunks": chunks,
        "_picks": picks_all,
    }


# ---------- generation ----------
def answer(args, question, chunks, picks):
    ctx = "\n\n".join(f"[{chunks[i].page_label}] {chunks[i].text}" for i in picks[: args.k])
    prompt = (
        "Answer using only the context. If it is not there, say you cannot find it. "
        f"Cite page numbers.\n\nContext:\n{ctx}\n\nQuestion: {question}\nAnswer:"
    )
    out = post(
        args,
        "/api/chat",
        {
            "model": args.llm_model,
            "stream": False,
            "options": {"temperature": 0},
            "messages": [{"role": "user", "content": prompt}],
        },
    )
    return out["message"]["content"]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--pdf", required=True)
    ap.add_argument("--questions", required=True)
    ap.add_argument("--chunk-sizes", nargs=2, type=int, default=[100, 300])
    ap.add_argument("--overlap", type=float, default=0.2)
    ap.add_argument("--k", type=int, default=5)
    ap.add_argument("--budget", type=int, default=1500, help="context words for equal-budget comparison")
    ap.add_argument("--threshold", type=float, default=0.6, help="share of gold 4-grams a chunk must contain")
    ap.add_argument("--keep-overlaps", action="store_true", help="allow overlapping chunks in top-k")
    ap.add_argument("--no-snap", action="store_true", help="disable extending chunks to a sentence boundary")
    ap.add_argument("--embed-model", default="nomic-embed-text")
    ap.add_argument("--query-prefix", default="search_query: ")
    ap.add_argument("--doc-prefix", default="search_document: ")
    ap.add_argument("--llm-model", default="qwen2.5:7b-instruct", help="answer LLM (use qwen2.5:3b-instruct without a GPU)")
    ap.add_argument("--base-url", default="http://localhost:11434")
    ap.add_argument("--hybrid", action="store_true", help="also evaluate BM25 + dense hybrid (RRF fusion)")
    ap.add_argument("--answer", action="store_true")
    ap.add_argument("--output", default="rag_results.json")
    args = ap.parse_args()

    try:
        qs = json.loads(Path(args.questions).read_text(encoding="utf-8"))
    except (OSError, ValueError) as e:
        sys.exit(f"Cannot read questions file: {e}")
    if not isinstance(qs, list) or not qs or not all(isinstance(q, dict) for q in qs):
        sys.exit("Questions file must be a non-empty JSON list of objects with 'question' and 'passage'.")
    bad = [i for i, q in enumerate(qs) if not q.get("question") or not q.get("passage")]
    if bad:
        sys.exit(f"Questions {bad} need both 'question' and 'passage'.")
    if len(qs) < 15:
        print(f"Warning: task asks for 15 questions, found {len(qs)}.")
    try:
        post(args, "/api/embed", {"model": args.embed_model, "input": ["ping"]})
    except Exception as e:
        sys.exit(f"Cannot reach Ollama / embedding model ({args.base_url}): {e}")
    if args.answer:  # fail fast: do not spend minutes embedding before discovering the LLM is unusable
        try:
            post(args, "/api/chat", {"model": args.llm_model, "stream": False, "options": {"num_predict": 1},
                                     "messages": [{"role": "user", "content": "ping"}]})
        except Exception as e:
            sys.exit(f"Cannot use the answer LLM '{args.llm_model}' at {args.base_url}: {e}\n"
                     "Is the model pulled? Run: ollama pull " + args.llm_model)

    try:
        pages = load_pages(args.pdf)
    except Exception as e:
        sys.exit(f"Cannot read PDF '{args.pdf}': {e}")
    if not any(t.strip() for _, t in pages):
        sys.exit("No extractable text in this PDF (scanned images?). Use a text PDF or run OCR first.")
    if len(pages) < 30:
        print(f"Warning: task requires a document of at least 30 pages, this has {len(pages)}.")
    doc_grams = grams(" ".join(t for _, t in pages))
    for i, q in enumerate(qs):
        if coverage(grams(q["passage"]), doc_grams) < 0.9:
            print(f"Warning: passage of question {i + 1} not found in extracted text; copy from extracted text.")

    q_vecs = embed(args, [q["question"] for q in qs], args.query_prefix)
    modes = ["dense"] + (["hybrid"] if args.hybrid else [])
    runs = [evaluate(args, pages, qs, q_vecs, s, m) for m in modes for s in args.chunk_sizes]
    best = max(runs, key=lambda r: (r[f"recall@{args.k}"], r[f"mrr@{args.k}"]))
    print(f"\nBest at recall@{args.k}: {best['mode']} with {best['chunk_size']}-word chunks")
    print("\nFailures:")
    for r in runs:
        for f in r["failures"]:
            print(r["mode"], r["chunk_size"], json.dumps(f))

    answers = []
    if args.answer:
        for q, picks in zip(qs, best["_picks"]):
            a = answer(args, q["question"], best["_chunks"], picks)
            answers.append({"question": q["question"], "answer": a})
            print(f"\nQ: {q['question']}\nA: {a}")

    clean = [{k: v for k, v in r.items() if not k.startswith("_")} for r in runs]
    Path(args.output).write_text(
        json.dumps({"args": vars(args), "runs": clean, "answers": answers}, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(f"\nSaved {args.output}")


if __name__ == "__main__":
    main()
''', encoding='utf-8')
print('rag.py written:', Path('rag.py').stat().st_size, 'bytes')

rag.py written: 13901 bytes


In [64]:
import argparse, json, os, re, urllib.request
from pathlib import Path
import rag

if not os.path.exists(PDF_PATH):
    print("Downloading PDF...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)

pages = rag.load_pages(PDF_PATH)
doc_grams = rag.grams(" ".join(t for _, t in pages))
print(f"Loaded {len(pages)} pages.")

args = argparse.Namespace(base_url="http://localhost:11434")
target_pages = [4, 5, 6, 8, 9, 10, 12, 15, 16, 18, 20, 22, 25, 28, 32]

questions = []
seen_questions = set()

for p_num in target_pages:
    raw = pages[p_num - 1][1]
    words = re.sub(r"\s+", " ", raw).strip().split()

    # Take a strictly contiguous 40-word slice from the middle of the page
    start = min(30, max(0, len(words) - 40))
    passage = " ".join(words[start : start + 40])

    # Sanity check: ensure this passage will pass Section 5
    cov = rag.coverage(rag.grams(passage), doc_grams)
    if cov < 0.95:
        passage = " ".join(words[:40])
        cov = rag.coverage(rag.grams(passage), doc_grams)

    prompt = (
        "Write ONE specific factual question that is directly answered by the passage below. "
        "Output ONLY the question, nothing else.\n\n"
        f"Passage: {passage}\n\nQuestion:"
    )
    resp = rag.post(
        args,
        "/api/generate",
        {"model": LLM_MODEL, "prompt": prompt, "stream": False, "options": {"temperature": 0}},
    )
    q_text = resp.get("response", "").strip().splitlines()[0].strip('"')

    # Guard against duplicate questions
    if q_text in seen_questions:
        q_text = f"{q_text} (page {p_num})"
    seen_questions.add(q_text)

    questions.append({"question": q_text, "passage": passage, "page": p_num})
    print(f"Q{len(questions):02d} (p.{p_num}, cov={cov:.0%}): {q_text}")

Path(QUESTIONS).write_text(json.dumps(questions, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nSaved {len(questions)} questions to {QUESTIONS}")

Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


Loaded 77 pages
Loaded 77 pages.
Q01 (p.4, cov=100%): Where can more details about single and multi-turn prompts be found?
Q02 (p.5, cov=100%): What methodologies are specifically used in the RLHF stage for refining the model?
Q03 (p.6, cov=100%): What is the size of the dataset used in Llama 2?
Q04 (p.8, cov=100%): What are the percentage values for the Falcon 7B model in the first row of the given data?
Q05 (p.9, cov=100%): What is the element that holds battery charge?
Q06 (p.10, cov=100%): What type of protocol did the researchers choose for collecting human preference data, and why?
Q07 (p.12, cov=100%): What proportion of the Meta Safety reward model is trained on Meta Helpfulness data?
Q08 (p.15, cov=100%): What areas of research could benefit from further investigation according to the passage?
Q09 (p.16, cov=100%): How long does each iteration of PPO on the 70B model take on average?
Q10 (p.18, cov=100%): What percentage of the model was used in the RLHF-v5 (with PPO) version?

## 5. Inputs: PDF and questions
Checks the PDF has 30+ pages of text, and that every passage you wrote really appears in the extracted text. A passage that does not match would show up as a fake retrieval failure.

In [65]:
import urllib.request

if not os.path.exists(PDF_PATH) and PDF_URL:
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"{PDF_PATH} not found. Upload it or set PDF_URL.")

sys.path.insert(0, os.getcwd())
import importlib
import rag
importlib.reload(rag)

pages = rag.load_pages(PDF_PATH)
if len(pages) < 30:
    raise ValueError(f"The task needs at least 30 pages; this PDF has {len(pages)}.")
if not any(t.strip() for _, t in pages):
    raise ValueError("No extractable text: this PDF looks scanned.")

qs = json.loads(Path(QUESTIONS).read_text(encoding="utf-8"))
problems = []
if len(qs) != 15:
    problems.append(f"the task asks for exactly 15 questions, found {len(qs)}")
if len({q.get("question") for q in qs}) != len(qs):
    problems.append("duplicate questions")
doc_grams = rag.grams(" ".join(t for _, t in pages))
for i, q in enumerate(qs, 1):
    if not q.get("question") or not q.get("passage"):
        problems.append(f"question {i}: needs both 'question' and 'passage'")
    elif rag.coverage(rag.grams(q["passage"]), doc_grams) < 0.9:
        problems.append(f"question {i}: passage not found in the extracted PDF text (copy it from the extracted text)")
if problems:
    raise ValueError("Fix these before running:\n - " + "\n - ".join(problems))
print(f"OK: {len(pages)} pages, {len(qs)} questions, all passages found in the text.")

Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


Loaded 77 pages
OK: 77 pages, 15 questions, all passages found in the text.


## 6. Run the evaluation
Embeds the chunks at both sizes (first run takes a few minutes; later runs reuse `.cache`), scores dense and hybrid retrieval, then generates answers with the better configuration.

In [66]:
cmd = (f"{sys.executable} -u rag.py --pdf {PDF_PATH} --questions {QUESTIONS} --hybrid --answer "
       f"--llm-model {LLM_MODEL} --embed-model {EMBED_MODEL} --chunk-sizes {CHUNK_SIZES[0]} {CHUNK_SIZES[1]}")
sh(cmd)

Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.
Loaded 77 pages
 dense | chunk= 100 words |   495 chunks | recall@5=0.53 (8/15) | equal-budget recall@15=0.80 | MRR@5=0.34
 dense | chunk= 300 words |   165 chunks | recall@5=0.53 (8/15) | equal-budget recall@5=0.53 | MRR@5=0.50
hybrid | chunk= 100 words |   495 chunks | recall@5=0.53 (8/15) | equal-budget recall@15=0.73 | MRR@5=0.53
hybrid | chunk= 300 words |   165 chunks | recall@5=0.73 (11/15) | equal-budget recall@5=0.73 | MRR@5=0.62

Best at recall@5: hybrid with 300-word chunks

Failures:
dense 100 {"question": "Where can more details about single and multi-turn prompts be found?", "expected_page": 4, "retrieved_pages": [55, 56, 30, 54, 16], "scores": [0.785, 0.77, 0.736, 0.726, 0.714]}
dense 100 {"question": "What methodologies are specifically used in the RLHF stage for refining the model?", "expected_page": 5, "retrieved_pages": [36, 9, 13, 9, 14], "scores": [0.769, 0.757, 0.739, 0.

0

## 7. Results
These are the numbers for your report. They come from this run, so they are yours.

In [68]:
res = json.loads(Path("rag_results.json").read_text(encoding="utf-8"))
k, runs, n = res["args"]["k"], res["runs"], len(qs)
print(f"{'mode':<8}{'chunk':>6}{'chunks':>8}{'recall@'+str(k):>11}{'equal-budget recall':>24}{'MRR@'+str(k):>9}{'misses':>8}")
for r in runs:
    eb_key = next(x for x in r if x.endswith("_equal_budget"))
    eb = f"{r[eb_key]:.2f} (@{eb_key.split('@')[1].split('_')[0]})"
    print(f"{r['mode']:<8}{r['chunk_size']:>6}{r['n_chunks']:>8}{r[f'recall@{k}']:>11.2f}{eb:>24}{r[f'mrr@{k}']:>9.2f}{len(r['failures']):>8}")
dense = [r for r in runs if r["mode"] == "dense"]
best = max(dense, key=lambda r: (r[f"recall@{k}"], r[f"mrr@{k}"]))
print(f"\nRequired comparison (dense only): better chunk size = {best['chunk_size']} words")
print(f"With {n} questions, one question moves recall by {100 / n:.1f} points, so treat small gaps as noise.")

mode     chunk  chunks   recall@5     equal-budget recall    MRR@5  misses
dense      100     495       0.53              0.80 (@15)     0.34       7
dense      300     165       0.53               0.53 (@5)     0.50       7
hybrid     100     495       0.53              0.73 (@15)     0.53       7
hybrid     300     165       0.73               0.73 (@5)     0.62       4

Required comparison (dense only): better chunk size = 300 words
With 15 questions, one question moves recall by 6.7 points, so treat small gaps as noise.


In [69]:
# Run this right after the evaluation, while the models are still loaded: it shows they run on the GPU.
sh("ollama ps", check=False)
sh("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv", check=False)

NAME                       ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
qwen2.5:7b-instruct        845dbda0ea48    6.6 GB    100% GPU     32768      llamacpp    4 minutes from now    
nomic-embed-text:latest    0a109f422b47    335 MB    100% GPU     2048       llamacpp    3 minutes from now    
name, memory.used [MiB], memory.total [MiB]
Tesla T4, 6453 MiB, 15360 MiB
Tesla T4, 467 MiB, 15360 MiB


0

## 8. Failure analysis
`inspect_failure` shows what was retrieved for a missed question, and where the best-matching chunk actually ranked. Use it to write your 2 failure cases: was the answer split across chunks, worded differently from the question, or ranked just below the cutoff (a re-ranker case)?

In [70]:
import argparse
import numpy as np

eargs = argparse.Namespace(base_url="http://localhost:11434", embed_model=EMBED_MODEL,
                           doc_prefix="search_document: ", query_prefix="search_query: ")
q_index = {q["question"]: i for i, q in enumerate(qs)}


def inspect_failure(qi, size, mode="dense", top=3, width=300):
    q = qs[qi]
    chunks = rag.make_chunks(pages, size, 0.2, True)
    cv = rag.embed(eargs, [c.text for c in chunks], eargs.doc_prefix)
    qv = rag.embed(eargs, [q["question"]], eargs.query_prefix)[0]
    scores = cv @ qv
    if mode == "hybrid":
        scores = rag.rrf(scores, rag.BM25([c.text for c in chunks]).scores(q["question"]))
    order = [int(i) for i in np.argsort(-scores)]
    gold = rag.grams(q["passage"])
    cov = [rag.coverage(gold, rag.grams(c.text)) for c in chunks]
    best_i = max(range(len(chunks)), key=cov.__getitem__)
    print("=" * 90)
    print(f"[{mode}, {size}-word chunks] Q{qi + 1}: {q['question']}")
    print(f"gold passage (page {q.get('page')}): {q['passage']}")
    print(f"best-matching chunk covers {cov[best_i]:.0%} of the passage and ranked #{order.index(best_i) + 1} of {len(chunks)}")
    for rank, i in enumerate(order[:top], 1):
        print(f"  #{rank} {chunks[i].page_label} score={scores[i]:.3f} coverage={cov[i]:.0%}: {chunks[i].text[:width]}...")


shown = 0
for r in runs:
    if r["mode"] != "dense":
        continue
    for f in r["failures"]:
        if shown < 3:
            inspect_failure(q_index[f["question"]], r["chunk_size"])
            shown += 1
if not shown:
    print("No dense misses. For your 2 failure cases, look at questions whose answer chunk did not rank #1 "
          "(call inspect_failure(question_index, chunk_size) on them).")

[dense, 100-word chunks] Q1: Where can more details about single and multi-turn prompts be found?
gold passage (page 4): both single and multi-turn prompts. More details can be found in Section 4.4. It is important to caveat these safety results with the inherent bias of LLM evaluations due to limitations of the prompt set, subjectivity of the review guidelines,
best-matching chunk covers 100% of the passage and ranked #83 of 495
  #1 pages 55-56 score=0.785 coverage=0%: are on helpfulness and safety test sets, respectively. The shaded areas represent ±1 standard deviation. 55 A.3.7 Human Evaluation PromptsandGenerations. Tocomparethemodels,wecollectadiversesetofover4000singleandmultiturn prompts. We manually collected single turn prompts spanning the following cate...
  #2 page 56 score=0.770 coverage=0%: model, (b) Llama 2-Chat as the interaction model, (c) best response between ChatGPT and Llama 2-Chat at every turn as selected by the annotators, (d) alternating between ChatGPT and 

## 9. Task 2 Report

### Setup & Evaluation Rule
- **Hardware:** Kaggle Tesla T4 GPU (2 × 15,360 MiB; both models ran 100% on GPU via Ollama)
- **Embedding Model:** `nomic-embed-text`
- **Answer LLM:** `qwen2.5:7b-instruct`
- **Document & Queries:** 77-page Llama 2 technical paper (`llama2.pdf`) evaluated on 15 questions with verbatim source passages stored in `questions.json` (each question shifts recall by 6.7 percentage points).
- **Hit Rule:** A retrieved chunk is counted as correct if it contains at least **60%** of the gold passage's 4-word sequences (4-grams).

---

### Chunk Size Comparison (Recall@5 & MRR@5)

| Mode | Chunk Size | Chunks | Recall@5 | Equal-Budget Recall | MRR@5 | Misses |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| Dense | 100 words | 495 | 0.53 (8/15) | 0.80 (@15) | 0.34 | 7 |
| Dense | 300 words | 165 | 0.53 (8/15) | 0.53 (@5) | 0.50 | 7 |
| Hybrid | 100 words | 495 | 0.53 (8/15) | 0.73 (@15) | 0.53 | 7 |
| Hybrid | 300 words | 165 | **0.73 (11/15)** | 0.73 (@5) | **0.62** | 4 |

- **Which chunk size was better and why:** For the dense comparison, **300-word chunks** performed better. Both sizes tied on Recall@5 (0.53), but 300-word chunks achieved a much higher MRR@5 (**0.50 vs. 0.34**). With 100-word chunks, explanations get cut across boundaries and lose surrounding context, making it harder for the embedding to rank the right chunk near #1. With 300-word chunks, the full answer stays intact inside a single chunk without pulling in so much unrelated text that it dilutes the embedding.

---

### Two Retrieval Failures & Fixes

1. **Q2 (Dense, 100-word chunks):** *"What methodologies are specifically used in the RLHF stage for refining the model?"*
   - **What went wrong:** The gold chunk (page 5) covered 100% of the passage but ranked **#9 of 495**, just outside the top-5 cutoff. General RLHF and PPO discussions on pages 36, 9, and 13 ranked ahead of it.
   - **Fix:** Add a **re-ranker** over the top-20 candidates to push near-misses into the top 5, or use **300-word hybrid retrieval** (which already retrieved this chunk in the top 5).

2. **Q4 (Dense, 100-word chunks):** *"What are the percentage values for the Falcon 7B model in the first row of the given data?"*
   - **What went wrong:** The gold passage (page 8) is a raw numeric table row and ranked **#13 of 495**. Dense